# The Technical Agent (The "Chartist")
This notebook implements a Random Forest Regressor to identify chart patterns and momentum shifts. 

First, we will load the engineered dataset we created in the data labeler from the `csv-history` folder.

In [3]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# Load the labeled data from the history folder
df = pd.read_csv("../csv-history/technical_data_prepared.csv", index_col=0, parse_dates=True)
df.head()

,Close,High,Low,Open,Volume,SMA_10,SMA_50,Daily_Return,Volatility,RSI,Target_5d_Return
Date,,,,,,,,,,,
2020-03-13,246.122818,248.096775,227.114370,240.429413,329566100,262.878595,291.763203,0.085486,0.043268,34.811180,-0.145457
2020-03-16,219.191147,234.772578,216.915612,220.406578,297240000,256.550978,290.209261,-0.109424,0.048475,31.285322,-0.064995
2020-03-17,231.025711,234.105454,216.650579,223.934090,262070500,252.215585,288.936976,0.053992,0.050879,35.773124,-0.032517
2020-03-18,219.328232,226.977299,208.380101,215.901228,327597100,245.557149,287.408258,-0.050633,0.051367,35.980682,0.034338
2020-03-19,219.794296,226.072576,212.218340,218.642828,289322000,239.895737,285.905493,0.002125,0.051460,36.360503,0.092412


Next, we separate our features from the target variable and split the data into training and testing sets. Because this is time-series data, we set `shuffle=False` to ensure we don't accidentally look into the future during training.

After splitting, we initialize and train the Random Forest model.

In [4]:
# Define Features (X) and Target (y)
features = ['Open', 'High', 'Low', 'Close', 'Volume', 'SMA_10', 'SMA_50', 'Volatility', 'RSI']
X = df[features]
y = df['Target_5d_Return']

# Split the data (No shuffling for time-series)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False)

# Train the Random Forest Agent
rf_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
print("Training the Technical Agent...")
rf_model.fit(X_train, y_train)
print("Training complete!")

Training the Technical Agent...
Training complete!


Now we evaluate how well the model learned by predicting on our test set. 

Finally, we simulate generating a "Momentum Score" for today by feeding the model the latest unlabelled data from our live dataset.

In [5]:
# Evaluate the model
y_pred = rf_model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse:.6f}")
print(f"R-squared Score: {r2:.4f}")

# Test a prediction using the live data (The "Momentum Score")
live_df = pd.read_csv("../csv-history/technical_data_live.csv", index_col=0, parse_dates=True)
latest_features = live_df[features].iloc[-1:] # Grab the most recent day

momentum_score = rf_model.predict(latest_features)[0]

print(f"\nCalculated Momentum Score for today: {momentum_score:.4f}")
if momentum_score > 0:
    print("Agent Signal: Positive Expected Momentum")
else:
    print("Agent Signal: Negative Expected Momentum")

Mean Squared Error: 0.001129
R-squared Score: -0.0044

Calculated Momentum Score for today: 0.0040
Agent Signal: Positive Expected Momentum
